In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
!pip install transformers accelerate bitsandbytes peft trl -q
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0

In [3]:
import os
import pandas as pd 
from kaggle_secrets import UserSecretsClient 

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

print("HF_TOKEN is loaded : ", HF_TOKEN is not None)

HF_TOKEN is loaded :  True


In [4]:
from huggingface_hub import login 
import pandas as pd 

login(token = HF_TOKEN)

df = pd.read_csv('/kaggle/input/datasets/pravargarg123/anemia-detection-medgamma/master.csv', keep_default_na = False)

train_df = df[df['split'] == 'train'].reset_index(drop = True)
val_df = df[df['split'] == 'valid'].reset_index(drop = True)

print('The shape of the training split :', train_df.shape)
print('The shape of the validation split : ', val_df.shape)
print(df['severity'].value_counts())

The shape of the training split : (66, 5)
The shape of the validation split :  (14, 5)
severity
None        39
Moderate    38
Mild        16
Severe       2
Name: count, dtype: int64


In [5]:
# Loading the MedGamma model from HuggingFace

from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_4bit = True)

model_id = 'google/medgemma-4b-it'

processor = AutoProcessor.from_pretrained(model_id, token = HF_TOKEN)

model = AutoModelForImageTextToText.from_pretrained(model_id, quantization_config = bnb_config, device_map = {"":0}, token = HF_TOKEN)


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

In [6]:
from peft import LoraConfig, get_peft_model 

lora_config = LoraConfig(r = 16, target_modules = ["q_proj", "v_proj"], lora_alpha = 32, lora_dropout = 0.05, task_type = "CAUSAL_LM")

lora_model = get_peft_model(model,lora_config)

lora_model.print_trainable_parameters()

trainable params: 6,447,104 || all params: 4,306,526,576 || trainable%: 0.1497


In [7]:
from datasets import Dataset
from PIL import Image 

def prepare_example(row):
    severity = row['severity']
    number = row['Number']
    image_row = Image.open(f"/kaggle/input/datasets/pravargarg123/anemia-detection-medgamma/images/{number}.jpg")
    prompt = 'This is an image of a patient with palpebral conjuctiva. Classify the anemia severity with one of the following: None, Mild, Moderate, Severe. Please give your answer in one word'

    message = [
        {
            "role": "user",
            "content" : [
                {"type": "image", "image" : image_row},
                {"type": "text", "text" : prompt}
            ]
        },
        {
            "role": "assistant",
            "content": [
                {"type" : "text", "text" : severity}
            ]
        }
    ]

    return {"message": message}

train_examples = [prepare_example(row) for _,row in train_df.iterrows()]
train_dataset = Dataset.from_list(train_examples)
print(train_dataset)
print(train_dataset[0])

validation_examples = [prepare_example(row) for _, row in val_df.iterrows()]
validation_dataset = Dataset.from_list(validation_examples)
print(validation_dataset)
print(validation_dataset[0])

Dataset({
    features: ['message'],
    num_rows: 66
})
{'message': [{'role': 'user', 'content': [{'type': 'image', 'image': {'path': '/kaggle/input/datasets/pravargarg123/anemia-detection-medgamma/images/41.jpg', 'bytes': None}, 'text': None}, {'type': 'text', 'image': None, 'text': 'This is an image of a patient with palpebral conjuctiva. Classify the anemia severity with one of the following: None, Mild, Moderate, Severe. Please give your answer in one word'}]}, {'role': 'assistant', 'content': [{'type': 'text', 'image': None, 'text': 'Mild'}]}]}
Dataset({
    features: ['message'],
    num_rows: 14
})
{'message': [{'role': 'user', 'content': [{'type': 'image', 'image': {'path': '/kaggle/input/datasets/pravargarg123/anemia-detection-medgamma/images/89.jpg', 'bytes': None}, 'text': None}, {'type': 'text', 'image': None, 'text': 'This is an image of a patient with palpebral conjuctiva. Classify the anemia severity with one of the following: None, Mild, Moderate, Severe. Please give y

In [8]:

if 'message' in train_dataset.column_names: 
    train_dataset = train_dataset.rename_column("message", "messages")
if 'message' in validation_dataset.column_names: 
    validation_dataset = validation_dataset.rename_column("message", "messages")

def collate_fn(examples):
   # print('collate_fn function is called, type : ', type(examples))
    if isinstance(examples, dict):
        examples = [dict(zip(examples.keys(), values)) for values in zip(*examples.values())]
        
    
    texts = []
    images = []

    for ex in examples:
        messages = ex['messages']  # list of [user_turn, assistant_turn]

        # find the user turn's content list, then pull out the image item from it
        user_content = messages[0]['content']  # user is first turn
        image_item = next(item for item in user_content if item['type'] == 'image')
        img = image_item['image']

        # the "image" field went through datasets' automatic Image feature casting,
        # so by the time we read it back here it's already a decoded PIL.Image object
        # (not the raw dict with a 'path' key that prepare_example originally created)
        if isinstance(img, dict):
            # fallback in case it ever comes back as a path/bytes dict instead
            img = Image.open(img['path']) if img.get('path') else Image.open(io.BytesIO(img['bytes']))
        images.append([img.convert("RGB")])

        # build the FULL chat-formatted text (includes user prompt + assistant answer)
        # this is what actually gets tokenized and fed to the model
        text = processor.apply_chat_template(messages, tokenize=False)
        texts.append(text)

    # batch-tokenize everything at once; processor handles padding across the batch
    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    # labels start as a copy of input_ids
    labels = batch["input_ids"].clone()

    # mask padding tokens so they don't contribute to loss
    labels[batch["attention_mask"] == 0] = -100

    # mask out everything except the assistant's answer:
    # find where the assistant's turn starts for each example, mask everything before it
    for i, text in enumerate(texts):
        # locate the assistant turn's text within the full formatted string
        assistant_text = "".join(
            c['text'] for c in examples[i]['messages'][1]['content'] if c['type'] == 'text'
        )
        # find token position where assistant's answer begins
        prefix = text.split(assistant_text)[0]  # everything before the answer
        prefix_len = len(processor.tokenizer(prefix, add_special_tokens=False)["input_ids"])
        labels[i, :prefix_len] = -100

    # this was missing before: attach labels to the batch and actually return it
    batch["labels"] = labels
    return batch


In [9]:
from trl import SFTConfig, SFTTrainer

training_arg = SFTConfig(
    output_dir="./medgemma-anemia-lora",
    num_train_epochs=3,
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size = 1, 
    gradient_accumulation_steps=4,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    remove_unused_columns= False,  
    dataset_kwargs = {"skip_prepare_dataset":True}
)

trainer = SFTTrainer(
    model=lora_model,
    args=training_arg,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=processor,
    data_collator=collate_fn  
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,8.734303,7.003405,8.299169,20724.000000,0.091083
2,4.085880,3.826937,3.799864,41448.000000,0.119999
3,3.563958,3.537398,3.736775,62172.000000,0.936295


TrainOutput(global_step=51, training_loss=6.582151216619155, metrics={'train_runtime': 3104.5253, 'train_samples_per_second': 0.064, 'train_steps_per_second': 0.016, 'total_flos': 1354313380768128.0, 'train_loss': 6.582151216619155, 'epoch': 3.0})

In [10]:
lora_model.save_pretrained('/kaggle/working/medgemma-anemia-lora')
processor.save_pretrained('/kaggle/working/medgemma-anemia-lora')

['/kaggle/working/medgemma-anemia-lora/processor_config.json']

In [11]:
import os
os.listdir('/kaggle/working/medgemma-anemia-lora')

['tokenizer_config.json',
 'checkpoint-17',
 'tokenizer.json',
 'checkpoint-51',
 'chat_template.jinja',
 'adapter_config.json',
 'README.md',
 'checkpoint-34',
 'adapter_model.safetensors',
 'processor_config.json']